---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [11]:
import os

!apt-get update -qq > /dev/null 2>&1
!apt-get install openjdk-11-jdk-headless -qq > /dev/null 2>&1

!pip install pyspark==3.5.1 -qq

java_home_path = "/usr/lib/jvm/java-11-openjdk-amd64"

if not os.path.exists(java_home_path):
    print("BŁĄD")
else:
    os.environ["JAVA_HOME"] = java_home_path

    from pyspark.sql import SparkSession
    try:
        spark = SparkSession.builder \
            .master("local[*]") \
            .appName("OtomotoBulletproof") \
            .config("spark.driver.memory", "2g") \
            .getOrCreate()

        print("Spark Session Created Successfully")
    except Exception as e:
        print(f"{e}")

Spark Session Created Successfully


---
# Zadanie samodzielne: Analiza Przestępczości w Chicago

Poniżej znajduje się miejsce na realizację zadania z analizy danych przy użyciu PySpark na zbiorze *Chicago Crimes* (około 50 000 ostatnich zdarzeń). Twoim celem jest przygotowanie, wyczyszczenie oraz zanalizowanie tych danych z wykorzystaniem zaawansowanych optymalizacji dostępnych w Sparku.

### Wymagania:
1. **Wczytanie i Czyszczenie Danych:** Wczytaj pobrany plik `chicago_crimes_sample.csv`. Usuń ewentualne duplikaty, wiersze z brakami danych (szczególnie w kluczowych kolumnach) i odfiltruj/napraw błędne daty.
2. **UDF i Pora Dnia:** Dodaj nową kolumnę z klasyfikacją pory dnia (np. noc, dzień, wieczór) utworzoną za pomocą User Defined Function (UDF) w oparciu o godzinę z kolumny `Date`.
3. **Optymalizacja i Partycjonowanie:** Zoptymalizuj przetwarzanie. Zastanów się, w których momentach użyć `cache()`. Przy dołączaniu mniejszych tabel słownikowych (jeśli byś je tworzył), wykorzystaj *broadcast join*. Ostatecznie zapisz przefiltrowane dane do formatu **Parquet** z podziałem na partycje według roku (`Year`).
4. **Analiza i Plany Zapytań:** Przeprowadź analizę statystyczną przestępstw (np. jakiego typu przestępstwa są najpopularniejsze w konkretnych lokacjach, o konkretnym czasie). Wykorzystaj funkcję `.explain()` aby udokumentować plan zapytania Sparka dla najcięższej agregacji.
5. *(Opcjonalnie)* **Uczenie Maszynowe (MLlib):** Spróbuj zbudować i wytrenować prosty model wieloklasowy, przewidujący rodzaj przestępstwa (`Primary Type`) na podstawie innych atrybutów, jak lokacja, godzina, arrest itp.

In [3]:
df_crimes = spark.read.option("header", True).csv("chicago_crimes_sample.csv")
df_crimes.show(5)

+-------------+--------------+--------------------+--------------------+----+------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+
|           id|   case_number|                date|               block|iucr|primary_type|         description|location_description|arrest|domestic|beat|district|ward|community_area|fbi_code|x_coordinate|y_coordinate|year|          updated_on|    latitude|    longitude|location|
+-------------+--------------+--------------------+--------------------+----+------------+--------------------+--------------------+------+--------+----+--------+----+--------------+--------+------------+------------+----+--------------------+------------+-------------+--------+
|     14193193|      JK249780|2026-05-07T00:00:...|057XX S MASSASOIT...|0820|       THEFT|      $500 AND UNDER|              STREET| false|   false|0811|     00

In [4]:
from pyspark.sql.functions import col, to_timestamp

df_crimes = spark.read.option("header", True).option("inferSchema", True).csv("chicago_crimes_sample.csv")

print(f"Liczba wierszy przed czyszczeniem: {df_crimes.count()}")
df_clean = df_crimes.dropDuplicates(subset=["id"])

kluczowe_kolumny = ["id", "case_number", "date", "primary_type"]
df_clean = df_clean.dropna(subset=kluczowe_kolumny)

format_daty = "MM/dd/yyyy hh:mm:ss a"

df_clean = df_clean.withColumn(
    "parsed_date",
    to_timestamp(col("date"), format_daty)
)

df_clean = df_clean.filter(col("parsed_date").isNotNull())

df_clean = df_clean.drop("date").withColumnRenamed("parsed_date", "date")

print(f"Liczba wierszy po czyszczeniu: {df_clean.count()}")
df_clean.select("id", "case_number", "date", "primary_type").show(5, truncate=False)

Liczba wierszy przed czyszczeniem: 149848
Liczba wierszy po czyszczeniu: 50000
+--------+-----------+-------------------+-----------------+
|id      |case_number|date               |primary_type     |
+--------+-----------+-------------------+-----------------+
|14110309|JK148980   |2026-02-14 05:20:00|CRIMINAL TRESPASS|
|14110322|JK148978   |2026-02-14 05:00:00|CRIMINAL DAMAGE  |
|14110344|JK148988   |2026-02-14 06:10:00|ASSAULT          |
|14110352|JK148960   |2026-02-14 05:04:00|OTHER OFFENSE    |
|14110362|JK149031   |2026-02-14 07:13:00|CRIMINAL DAMAGE  |
+--------+-----------+-------------------+-----------------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import udf, col, hour
from pyspark.sql.types import StringType

def przypisz_pore_dnia(godzina):
    if godzina is None:
        return "Nieznana"

    if 6 <= godzina < 12:
        return "Rano"
    elif 12 <= godzina < 18:
        return "Dzień"
    elif 18 <= godzina < 22:
        return "Wieczor"
    else:
        return "Noc"
pora_dnia_udf = udf(przypisz_pore_dnia, StringType())

df_clean = df_clean.withColumn(
    "pora_dnia",
    pora_dnia_udf(hour(col("date")))
)

df_clean.select("date", "pora_dnia").show(10, truncate=False)

+-------------------+---------+
|date               |pora_dnia|
+-------------------+---------+
|2026-02-14 12:23:00|Dzień    |
|2026-02-14 20:49:00|Wieczor  |
|2026-02-14 22:05:00|Noc      |
|2026-02-15 03:00:00|Noc      |
|2026-02-15 14:00:00|Dzień    |
|2026-02-15 16:00:00|Dzień    |
|2026-02-15 21:39:00|Wieczor  |
|2026-02-15 16:00:00|Dzień    |
|2026-02-16 00:01:00|Noc      |
|2026-02-16 23:33:00|Noc      |
+-------------------+---------+
only showing top 10 rows



In [13]:
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StructType, StructField, StringType


df_clean.cache()
print(f"Liczba rekordów w cache: {df_clean.count()}")
slownik_dane = [
    ("THEFT", "Przestępstwo przeciwko mieniu"),
    ("BATTERY", "Przestępstwo przeciwko zdrowiu i życiu"),
    ("NARCOTICS", "Przestępstwa narkotykowe"),
    ("ROBBERY", "Przestępstwo przeciwko mieniu"),
    ("CRIMINAL DAMAGE", "Przestępstwo przeciwko mieniu")
]

schemat_slownika = StructType([
    StructField("primary_type", StringType(), True),
    StructField("kategoria", StringType(), True)
])

df_slownik = spark.createDataFrame(slownik_dane, schema=schemat_slownika)

df_optimized = df_clean.join(broadcast(df_slownik), on="primary_type", how="left")


sciezka_zapisu = "chicago_crimes_optimized.parquet"

df_optimized.write \
    .mode("overwrite") \
    .partitionBy("year") \
    .parquet(sciezka_zapisu)

print(f"Zapisano dane do katoalogu {sciezka_zapisu}")

Liczba rekordów w cache: 50000
Zapisano dane do katoalogu chicago_crimes_optimized.parquet


In [14]:
from pyspark.sql.functions import col, count, desc, row_number
from pyspark.sql.window import Window

df_grouped = df_optimized.groupBy("location_description", "pora_dnia", "primary_type") \
    .agg(count("*").alias("liczba_przestepstw"))

window_spec = Window.partitionBy("location_description", "pora_dnia") \
                    .orderBy(desc("liczba_przestepstw"))

df_top_crimes = df_grouped.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") == 1) \
    .drop("rank")

df_top_crimes.orderBy(desc("liczba_przestepstw")).show(10, truncate=False)


df_top_crimes.explain(mode="formatted")

+--------------------+---------+-------------------+------------------+
|location_description|pora_dnia|primary_type       |liczba_przestepstw|
+--------------------+---------+-------------------+------------------+
|STREET              |Noc      |MOTOR VEHICLE THEFT|1068              |
|APARTMENT           |Noc      |BATTERY            |1030              |
|STREET              |Wieczor  |MOTOR VEHICLE THEFT|771               |
|APARTMENT           |Dzień    |BATTERY            |744               |
|STREET              |Dzień    |MOTOR VEHICLE THEFT|627               |
|APARTMENT           |Wieczor  |BATTERY            |625               |
|APARTMENT           |Rano     |BATTERY            |620               |
|SMALL RETAIL STORE  |Dzień    |THEFT              |591               |
|STREET              |Rano     |THEFT              |506               |
|DEPARTMENT STORE    |Dzień    |THEFT              |503               |
+--------------------+---------+-------------------+------------

In [9]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# --- 0. Przygotowanie danych ---
cols_to_use = ["location_description", "pora_dnia", "arrest", "primary_type"]
df_ml = df_optimized.select(cols_to_use).dropna()

df_ml = df_ml.withColumn("arrest", col("arrest").cast("string"))

# --- 1. STRING INDEXER  ---
indexer_loc = StringIndexer(inputCol="location_description", outputCol="loc_idx", handleInvalid="keep")
indexer_pora = StringIndexer(inputCol="pora_dnia", outputCol="pora_idx", handleInvalid="keep")
indexer_arrest = StringIndexer(inputCol="arrest", outputCol="arrest_idx", handleInvalid="keep")

indexer_label = StringIndexer(inputCol="primary_type", outputCol="label", handleInvalid="skip")

# --- 2. VECTOR ASSEMBLER (Pakowanie cech) ---
assembler = VectorAssembler(
    inputCols=["loc_idx", "pora_idx", "arrest_idx"],
    outputCol="features"
)

# --- 3. INICJALIZACJA MODELU ---
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=20, maxDepth=5, maxBins=128)

# --- 4. PIPELINE  ---
pipeline = Pipeline(stages=[
    indexer_loc, indexer_pora, indexer_arrest, indexer_label, assembler, rf
])

# --- 5. PODZIAŁ DANYCH ---
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("Trenowanie modelu Random Forest")
# --- 6. TRENOWANIE ---
model = pipeline.fit(train_data)
print(" Model wytrenowany!")

# --- 7. PREDYKCJA NA DANYCH TESTOWYCH ---
predictions = model.transform(test_data)

# --- 8. EWALUACJA (Ocena jakości) ---
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Accuracy modelu: {accuracy:.2%}\n")

# --- 9. ODKODOWANIE WYNIKÓW I PODGLĄD ---
label_converter = IndexToString(inputCol="prediction", outputCol="predicted_type", labels=model.stages[3].labels)
predictions_text = label_converter.transform(predictions)

print("Tabela z przewidywaniami modelu na danych testowych:")
predictions_text.select("location_description", "pora_dnia", "arrest", "primary_type", "predicted_type").show(15, truncate=False)

Trenowanie modelu Random Forest
 Model wytrenowany!
Accuracy modelu: 31.45%

Tabela z przewidywaniami modelu na danych testowych:
+--------------------+---------+------+------------------+--------------+
|location_description|pora_dnia|arrest|primary_type      |predicted_type|
+--------------------+---------+------+------------------+--------------+
|ALLEY               |Noc      |false |CRIMINAL DAMAGE   |BATTERY       |
|ALLEY               |Wieczór  |false |DECEPTIVE PRACTICE|BATTERY       |
|APARTMENT           |Dzień    |false |ASSAULT           |BATTERY       |
|APARTMENT           |Dzień    |false |DECEPTIVE PRACTICE|BATTERY       |
|APARTMENT           |Dzień    |true  |BATTERY           |BATTERY       |
|APARTMENT           |Noc      |false |BATTERY           |BATTERY       |
|APARTMENT           |Noc      |false |CRIMINAL DAMAGE   |BATTERY       |
|APARTMENT           |Noc      |false |ROBBERY           |BATTERY       |
|APARTMENT           |Rano     |false |BATTERY          

In [10]:
from pyspark.sql.functions import dayofweek, month
from pyspark.ml.feature import StringIndexer, VectorAssembler, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

# --- 1. NOWE CECHY ---
df_ml_v2 = df_clean.withColumn("dzien_tygodnia", dayofweek(col("date"))) \
                   .withColumn("miesiac", month(col("date")))

cols_to_use_v2 = ["location_description", "pora_dnia", "arrest", "district", "dzien_tygodnia", "miesiac", "primary_type"]
df_ml_v2 = df_ml_v2.select(cols_to_use_v2).dropna()
df_ml_v2 = df_ml_v2.withColumn("arrest", col("arrest").cast("string"))


# --- 2. ZAWĘŻENIE PROBLEMU DO TOP 5 PRZESTĘPSTW ---
top_5_crimes = ["THEFT", "BATTERY", "CRIMINAL DAMAGE", "NARCOTICS", "ASSAULT"]
df_ml_v2 = df_ml_v2.filter(col("primary_type").isin(top_5_crimes))


# --- 3. STRING INDEXERS ---
indexer_loc = StringIndexer(inputCol="location_description", outputCol="loc_idx", handleInvalid="keep")
indexer_pora = StringIndexer(inputCol="pora_dnia", outputCol="pora_idx", handleInvalid="keep")
indexer_arrest = StringIndexer(inputCol="arrest", outputCol="arrest_idx", handleInvalid="keep")
indexer_dist = StringIndexer(inputCol="district", outputCol="dist_idx", handleInvalid="keep")
indexer_label = StringIndexer(inputCol="primary_type", outputCol="label", handleInvalid="skip")


# --- 4. ASSEMBLER  ---
assembler = VectorAssembler(
    inputCols=["loc_idx", "pora_idx", "arrest_idx", "dist_idx", "dzien_tygodnia", "miesiac"],
    outputCol="features"
)

# --- 5. MOCNIEJSZY MODEL ---
rf_v2 = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=30, maxDepth=10, maxBins=256)


# --- 6. TRENOWANIE I OCENA ---
pipeline_v2 = Pipeline(stages=[
    indexer_loc, indexer_pora, indexer_arrest, indexer_dist, indexer_label, assembler, rf_v2
])

train_data, test_data = df_ml_v2.randomSplit([0.8, 0.2], seed=42)
print("Trenowanie ulepszonego modelu")

model_v2 = pipeline_v2.fit(train_data)
predictions_v2 = model_v2.transform(test_data)

label_converter = IndexToString(inputCol="prediction", outputCol="predicted_type", labels=model_v2.stages[4].labels)
results = label_converter.transform(predictions_v2)

print("\n--- Próbka przewidywań ulepszonego modelu ---")
results.select("primary_type", "predicted_type", "probability").show(10, truncate=False)

Trenowanie ulepszonego modelu

--- Próbka przewidywań ulepszonego modelu ---
+---------------+--------------+------------------------------------------------------------------------------------------------------+
|primary_type   |predicted_type|probability                                                                                           |
+---------------+--------------+------------------------------------------------------------------------------------------------------+
|CRIMINAL DAMAGE|BATTERY       |[0.1709549559118809,0.363902610834155,0.3251772460403518,0.12218342429284856,0.017781762920763805]    |
|BATTERY        |BATTERY       |[0.27503868957734046,0.3675017642395102,0.1605069519299072,0.19170605415143532,0.005246540101806923]  |
|ASSAULT        |THEFT         |[0.5309460318296148,0.2567123655454607,0.11085692162887553,0.09841683100604133,0.003067849990007713]  |
|CRIMINAL DAMAGE|BATTERY       |[0.18870032432507647,0.4517550115087387,0.1950010697320398,0.16296418408161